# Low Fidelity Neural Network
Neural Network which uses LF dataset and exploits the Low fidelity component of the MF model. The NN is tested also on the HF test set  

In [1]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from keras.utils import custom_object_scope
from keras.initializers import glorot_uniform
from keras.models import load_model, save_model
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam, Nadam, Adamax
from ann_functions3D import (
    getModel,
    kCrossVal,
    transfBestparam,
    kCrossValSingle,
    import_data, normalization
)
from time import perf_counter
import pandas
import os
from itertools import product
import keras
import tensorflow as tf


In [2]:
# reproducibility
seed=42
np.random.seed(seed)

keras.utils.set_random_seed(seed)

tf.random.set_seed(seed)

# Data Preparation

In [3]:
file_path_LF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_LF_46_d75.mat")
(reaction_LF_test, U_LF_test, t_LF_test) = import_data(file_path_LF)
U_LF_test = U_LF_test[:, :, 12,12]
t_LF_test=t_LF_test.T

In [4]:
reaction_LF_test = normalization(reaction_LF_test)
t_LF_test = normalization(t_LF_test)
U_LF_test = normalization(U_LF_test)

In [5]:
NepoLF=1000
Nlf=200

In [6]:
# TRANSFORMATION

permutation1 = np.random.permutation(len(reaction_LF_test))
permutation2 = np.random.permutation(len(t_LF_test))

reaction_LF = reaction_LF_test[permutation1][0:Nlf]
t_LF = t_LF_test[permutation2][0:Nlf]

grid1, grid2 = np.meshgrid(reaction_LF, t_LF)
reaction_LF = np.column_stack((grid1.ravel(), grid2.ravel()))
reaction_LF_test = np.array(list(product(reaction_LF_test.flatten(), t_LF_test.flatten())))

reaction_LF=np.c_[reaction_LF, np.abs(np.sin(3.5*np.pi*reaction_LF[:, 1])),np.ones(reaction_LF.shape[0])]
reaction_LF_test=np.c_[reaction_LF_test, np.abs(np.sin(3.5*np.pi*reaction_LF_test[:, 1])),np.ones(reaction_LF_test.shape[0])]

##
p1, p2 = np.meshgrid(permutation1[0:Nlf], permutation2[0:Nlf])
permutation = np.column_stack((p1.ravel(), p2.ravel()))
U_LF = U_LF_test[permutation[:,0],permutation[:,1]] 
row, col = U_LF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test = U_LF_test.flatten()[comb]
##

# Low Fidelity Neural Network
## Low fidelity data

In [7]:
# start = perf_counter()

# reaction_final = np.vstack(
#     (reaction_LF)
# )  

# ##########################       NN_LF     ##########################
# ####################    HYPERPARAMETER OPTIMIZATION    #######################
# MAX_EVAL = 2
# name = "LF"
# K.clear_session()
# bayes_trials = Trials()
# opt_list = ["Adam", "Adamax"]
# kernel_list = ["uniform", "glorot_uniform"]
# aux_dic = {"opt": opt_list, "kernel_init": kernel_list}
# space = {
#     "nodes": hp.qloguniform("nodes", np.log(4), np.log(64), 2),
#     "l2weight": hp.loguniform("l2weight", np.log(0.0001), np.log(100)),
#     "lr": hp.loguniform("lr", np.log(0.0001), np.log(0.1)),
#     "kernel_init": hp.choice("kernel_init", kernel_list),
#     "opt": hp.choice("opt", opt_list),
# }


# def objective(params):
#     K.clear_session()
#     CVres = kCrossVal(2,Nlf, NepoLF, reaction_final, U_LF, params, name)
#     return {"loss": CVres, "params": params, "status": STATUS_OK}


# best_params = fmin(
#     fn=objective,
#     space=space,
#     algo=tpe.suggest,
#     max_evals=MAX_EVAL,
#     trials=bayes_trials,
# )

# transfBestparam(best_params, aux_dic)
# print(best_params)

####################    NN training and PREDICTION    #######################


start = perf_counter()
name="LF"
K.clear_session()

best_params = {
                    "lr": 0.0255,
                    "kernel_init": "glorot_uniform",
                    "opt": "Adam",
                } 


finalModel = getModel(
    best_params, name
)  # final model chosen according to the best paramters
hist = finalModel.fit(
    reaction_LF,
    U_LF,
    validation_data=(reaction_LF_test, U_LF_test),
    epochs=NepoLF,
    batch_size=Nlf,
    verbose=0,
    validation_freq=20,
)

ULF = finalModel.predict(reaction_LF_test)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nLF Model:")

test_mse = np.mean(np.square(U_LF_test - ULF[:, 0]))
print(f"Test MSE: {test_mse}")

r2_LF = 1 - np.sum(np.square(U_LF_test - ULF[:, 0])) / np.sum(
    np.square(U_LF_test - np.mean(U_LF_test))
)
print(f"R^2: {r2_LF}")

1252/1252 [==============================] - 2s 2ms/step
Elapsed time:  219.2823654001113

LF Model:
Test MSE: 0.3673992424312155
R^2: -2.0332083401701975


# High fidelity data

In [8]:
########################     PREPARATION      ##########################
# introduction of the data
file_path_HF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_HF.mat")
(reaction_HF_test, U_HF_test, t_HF_test) = import_data(file_path_HF)
t_HF_test=t_HF_test.T
U_HF_test = U_HF_test[:, :, 44,44]

In [9]:
########################     NORMALIZATION  #########################
# Input

reaction_HF_test = normalization(reaction_HF_test)
t_HF_test=normalization(t_HF_test)
U_HF_test=normalization(U_HF_test)

In [10]:

reaction_HF_test = np.array(list(product(reaction_HF_test.flatten(), t_HF_test.flatten())))
reaction_HF_test=np.c_[reaction_HF_test, np.abs(np.sin(3.5*np.pi*reaction_HF_test[:, 1])),np.ones(reaction_HF_test.shape[0])]

row, col = U_HF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_HF_test = U_HF_test.flatten()[comb]


In [12]:
hist = finalModel.fit(
    reaction_LF,
    U_LF,
    validation_data=(reaction_HF_test, U_HF_test),
    epochs=NepoLF,
    batch_size=Nlf,
    verbose=0,
    validation_freq=20,
)

UHF = finalModel.predict(reaction_HF_test)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nHF Data")
print("\nLF Model:")


test_mse_HF = np.mean(np.square(U_HF_test - UHF[:, 0]))
print(f"Test MSE: {test_mse_HF}")

r2_HF = 1 - np.sum(np.square(U_HF_test - UHF[:, 0])) / np.sum(
    np.square(U_HF_test - np.mean(U_HF_test))
)
print(f"R^2: {r2_HF}")

1252/1252 [==============================] - 2s 2ms/step
Elapsed time:  652.2149617001414

HF Data

LF Model:
Test MSE: 0.37269667817925306
R^2: -2.308641727867254


# ADD PLOT

In [13]:
os.makedirs("NNLF_LF200_1000steps")

R2_HF = pandas.DataFrame({"R2_HF": [r2_HF]})
R2_LF = pandas.DataFrame({"R2_LF": [r2_LF]})
MSE_test_LF = pandas.DataFrame({"MSE_test_LF": [test_mse]})
MSE_test_HF = pandas.DataFrame({"MSE_test_HF": [test_mse_HF]})


R2_LF.to_csv(
    "./NNLF_LF200_1000steps/r2_HF.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
MSE_test_HF.to_csv(
    "./NNLF_LF200_1000steps/test_mse.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
R2_HF.to_csv(
    "./NNLF_LF200_1000steps/r2_LF_lhs.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)
MSE_test_LF.to_csv(
    "./NNLF_LF200_1000steps/mse_LF_lhs.txt",
    header=True,
    index=False,
    sep="\t",
    mode="a",
)